# <p style="text-align:center;">This Project Objective</p>
***In this notebook, I aim to train convolutional neural network (CNN) models using transfer learning. Given a small dataset, utilizing a pre-trained model like EfficientNetB7 is advantageous due to its ability to leverage knowledge learned from the ImageNet dataset, which consists of 1000 different classes. Transfer learning allows us to take advantage of features learned by the pre-trained model and adapt them to our specific classification problem, mitigating the data hunger typically associated with deep learning models. EfficientNetB7, being a powerful CNN architecture, offers a robust foundation for classification tasks, especially with limited data. By fine-tuning its weights on our dataset, we can achieve efficient and accurate predictions. Transfer learning with EfficientNetB7 is a prudent choice for classification tasks where datasets are small, as it harnesses the representation power of a large, diverse dataset like ImageNet while requiring fewer training examples. This approach not only enhances model performance but also optimizes computational resources, making it a suitable solution for various classification challenges.***

# <p style="text-align:center;">About Data</p>
***Who doesn't like pizza? This dataset contains about 1000 images of pizza and 1000 images of dishes other than pizza. It can be used for a simple binary image classification task.***

***All images were rescaled to have a maximum side length of 512 pixels.***

***This is a subset of the Food-101 dataset. Information about the original dataset can be found in the following paper:
Bossard, Lukas, Matthieu Guillaumin, and Luc Van Gool. "Food-101 – Mining Discriminative Components with Random Forests." In European conference on computer vision, pp. 446-461. Springer, Cham, 2014.***

***The original dataset can be found in the following locations:***
https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/
https://www.kaggle.com/datasets/dansbecker/food-101
https://paperswithcode.com/dataset/food-101
https://www.tensorflow.org/datasets/catalog/food101

***Number of instances in each class:***
***Pizza: 983***
***Not Pizza: 983***


# <p style="text-align:center;">Importing Libraries </p>

In [ ]:
#Import Os and Basis Libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 
#Matplot Images
import matplotlib.image as mpimg
#Tensflor and Keras Layer and Model and Optimize and Loss
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import *
from tensorflow.keras.losses import BinaryCrossentropy
import tensorflow_hub as hub
from tensorflow.keras.optimizers import Adam
#PreTrained Model VGG16
from keras.applications.vgg16 import VGG16
#Image Generator DataAugmentation
from keras.preprocessing.image import ImageDataGenerator, array_to_img, img_to_array, load_img
from tensorflow.keras.preprocessing import image
#Early Stopping
from tensorflow.keras.callbacks import EarlyStopping
#Warnings Remove 
import warnings 
warnings.filterwarnings("ignore")

# <p style="text-align:center;">Loading Data </p>

In [ ]:
# Directory containing the "Train" folder
directory = "/kaggle/input/pizza-not-pizza/"

# List of categories (subfolder names)
categories = ["not_pizza", "pizza"]

# Initialize lists to store filenames and categories
filenames = []
category_labels = []

# Iterate through the categories
for category in categories:
    # Path to the current category folder
    category_folder = os.path.join(directory, "pizza_not_pizza", category)
    # List all filenames in the category folder
    category_filenames = os.listdir(category_folder)
    # Append filenames and corresponding category labels
    filenames.extend(category_filenames)
    category_labels.extend([category] * len(category_filenames))

# Create DataFrame
df = pd.DataFrame({
    'filename': filenames,
    'category': category_labels
})

# Display the first few rows of the DataFrame
print(df.head())

In [ ]:
# Shape 
print(f'The Shape Of The Data is : {df.shape}')

# <p style="text-align:center;">Visualizing Data Count</p>

In [ ]:
# Count the occurrences of each category in the 'category' column
count = df['category'].value_counts()

# Pie chart using Seaborn
plt.figure(figsize=(6, 6))  # Set the size of the plot
sns.set_palette("pastel")  # Set color palette
plt.pie(count, labels=count.index, autopct='%1.1f%%', startangle=140)  # Create the pie chart
plt.title('Distribution of Categories')  # Add a title
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
plt.show()  # Show the plot


# <p style="text-align:center;">Visualizing Some Images</p>

## ***NotPizza***

In [ ]:
#Function to Visualize Images From Path 
def visualize_images(path, num_images=5):
    # Get a list of image filenames in the specified path
    image_filenames = os.listdir(path)
    
    # Limit the number of images to visualize if there are more than num_images
    num_images = min(num_images, len(image_filenames))
    
    # Create a figure and axis object to display images
    fig, axes = plt.subplots(1, num_images, figsize=(15, 3))
    
    # Iterate over the selected images and display them
    for i, image_filename in enumerate(image_filenames[:num_images]):
        # Load the image using Matplotlib
        image_path = os.path.join(path, image_filename)
        image = mpimg.imread(image_path)
        
        # Display the image
        axes[i].imshow(image)
        axes[i].axis('off')  # Turn off axis
        axes[i].set_title(image_filename)  # Set image filename as title
    
    # Adjust layout and display the figure
    plt.tight_layout()
    plt.show()

# No-Pizza image path
path_to_visualize = "/kaggle/input/pizza-not-pizza/pizza_not_pizza/not_pizza"  # Change this to the desired path

# Visualize some images from No-Pizza path
visualize_images(path_to_visualize, num_images=5)

## ***Pizza***

In [ ]:
#  Pizza Images Path
path_to_visualize = "/kaggle/input/pizza-not-pizza/pizza_not_pizza/pizza"  # Change this to the desired path

# Visualize some images from Pizza path
visualize_images(path_to_visualize, num_images=5)

# <p style="text-align:center;">Model Building</p>

###### **<h1 align="center">Data Augmentation</span>**

***Data augmentation is a technique used to generate additional data from a limited dataset. It plays a crucial role in machine learning and deep learning applications. By applying various transformations to the existing data, such as rotation, scaling, flipping, or adding noise, data augmentation effectively expands the dataset, thereby enhancing model performance and generalization.***

***When using flow_from_directory with data augmentation, the generator generates augmented images on-the-fly during training epochs. This means that the original images in your directory are not physically modified or increased in number; instead, the generator generates new augmented images each time a batch is requested during training.***

In [ ]:
#intilize Data Directory 
data_dir = '/kaggle/input/pizza-not-pizza/pizza_not_pizza'

#Training Augmented Data
# Defining data generator with Data Augmentation
data_gen_augmented = ImageDataGenerator(rescale = 1/255., 
                                        validation_split = 0.2,
                                        zoom_range = 0.2,
                                        horizontal_flip= True,
                                        rotation_range = 20,
                                        width_shift_range=0.2,
                                        height_shift_range=0.2)
print('Augmented training Images:')
train_generator = data_gen_augmented.flow_from_directory(data_dir, 
                                                              target_size = (224, 224), 
                                                              batch_size = 32,
                                                              subset = 'training',
                                                              class_mode = 'binary')

#Testing Augmented Data
# Defining Validation_generator withour Data Augmentation
data_gen = ImageDataGenerator(rescale = 1/255., validation_split = 0.2)

print('Unchanged Validation Images:')
validation_generator = data_gen.flow_from_directory(data_dir, 
                                        target_size = (224, 224), 
                                        batch_size = 32,
                                        subset = 'validation',
                                        class_mode = 'binary')

# <p style="text-align:center;">Transfer Learning</p>

***Transfer learning is a pivotal research paradigm in machine learning that leverages knowledge gained from solving one problem and applies it to a different yet related problem. This approach significantly streamlines the process of solving new tasks by capitalizing on previously learned patterns and representations. Inspired by the adaptive learning mechanisms observed in humans, transfer learning has emerged as a cornerstone in the machine learning industry.***

###### **<h1 align="center">EfficientNetB7 Base</span>**
***EfficientNet is a family of convolutional neural network models that have been developed with the goal of achieving better accuracy and efficiency compared to existing architectures. EfficientNetB7 is one of the larger variants in this family. The "B7" suffix indicates the size of the model, with larger numbers representing larger models.***

In [ ]:
#Using EfficientNetB7 
link_model = 'https://tfhub.dev/google/efficientnet/b7/feature-vector/1'

resnet_base = hub.KerasLayer(link_model,
                                trainable=False,
                                input_shape=(224, 224, 3))

###### **<h1 align="center">Dense Part</span>**

In [ ]:
model = Sequential()
model.add(resnet_base)
model.add(Dense(1,activation = 'sigmoid'))

###### **<h1 align="center">Compile Model</span>**

In [ ]:
model.compile(loss = BinaryCrossentropy(),
                optimizer = Adam(),
                metrics = ['accuracy'])

###### **<h1 align="center">Fitting Model</span>**

In [ ]:
history = model.fit(train_generator,
                        epochs= 10,
                        steps_per_epoch = len(train_generator),
                        validation_data = validation_generator,
                        validation_steps = len(validation_generator))

###### **<h1 align="center">Evaluate</span>**

In [ ]:
# Evaluate the model on the validation dataset
validation_loss, validation_accuracy = model.evaluate(validation_generator)

# Print the validation loss and accuracy
print("Validation Loss:", validation_loss)
print("Validation Accuracy:", validation_accuracy)

###### **<h1 align="center">Accuracy</span>**

In [ ]:
plt.plot(history.history['accuracy'],color='red',label='train')
plt.plot(history.history['val_accuracy'],color='blue',label='validation')
plt.legend()
plt.show()

###### **<h1 align="center">Loss</span>**

In [ ]:
plt.plot(history.history['loss'],color='red',label='train')
plt.plot(history.history['val_loss'],color='blue',label='validation')
plt.legend()
plt.show()

# <p style="text-align:center;">Predictions On Unseen Data</p>

In [ ]:
# List of paths to your single images
image_paths = ['/kaggle/input/pizza-images/Pizza not.jpeg', '/kaggle/input/pizza-images/pizza 1.jpg', '/kaggle/input/pizza-images/pizza.jpg']
true_labels = ['Pizza', 'No Pizza', 'No Pizza']

# Load and preprocess each image, make predictions, and display them
for img_path, true_label in zip(image_paths, true_labels):
    # Load and preprocess the image
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0  # Normalize pixel values

    # Make predictions
    predictions = model.predict(img_array)
    binary_prediction = (predictions > 0.5).astype(int)

    # Display the image with true and predicted labels
    plt.imshow(img)
    plt.axis('off')
    if binary_prediction[0][0] == 1:
        predicted_label = 'Pizza'
    else:
        predicted_label = 'Not Pizza'
    plt.title(f'Predicted: {predicted_label} (True: {true_label})')
    plt.show()

**Observation :**

***Our Model is Performing Well on Unseen Data***

# ***Conclusion*** 

***Transfer learning optimizes model training by leveraging pre-existing knowledge from a pre-trained model, such as EfficientNetB7, for specific tasks like Pizza vs No Pizza classification. Achieving 97% accuracy signifies effective learning. EfficientNetB7, trained on ImageNet, provides a robust foundation for classification tasks, particularly with limited data. This approach enhances efficiency by utilizing learned features, enabling accurate predictions while conserving computational resources. Leveraging transfer learning with EfficientNetB7 ensures efficient model training for Pizza vs No Pizza classification, yielding high accuracy and resource optimization.***